# Partial Least Squares Regression: Theory and Practice

## SOLUTIONS NOTEBOOK

**Course:** Machine Learning

---

> **This notebook contains complete solutions to all homework problems.**
> The textbook sections (Part 1) are identical to the student version.

## Part 0: Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

from utils import (
    generate_low_rank_data,
    generate_sparse_data,
    generate_collinear_data,
    mse,
    relative_mse,
)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
sns.set_style("whitegrid")
print("Setup complete.")

---
# Part 1: The Theory of Partial Least Squares Regression

*This part is a textbook — read it carefully. There are no problems to solve here.*

## 1.1 What Problem Does PLS Solve?

Consider the standard regression setup: we observe a matrix $\mathbf{X} \in \mathbb{R}^{n \times p}$ of features and a response vector $\mathbf{y} \in \mathbb{R}^n$, and we want to predict $\mathbf{y}$ from $\mathbf{X}$.

When $p \gg n$ (many more features than samples), ordinary least squares (OLS) fails: the system $\mathbf{X}\boldsymbol{\beta} = \mathbf{y}$ is underdetermined, and $\mathbf{X}^T\mathbf{X}$ is singular. Even when $p < n$, if features are highly collinear, OLS estimates are wildly unstable.

Two classical remedies are:

| Method | Strategy | Hyperparameter |
|--------|----------|----------------|
| **Ridge** | Shrink all coefficients toward zero | $\lambda$ (penalty strength) |
| **PLS** | Project onto $k$ latent directions that maximize covariance with $\mathbf{y}$ | $k$ (number of components) |

Both are forms of **regularization**, but they regularize in fundamentally different ways. This chapter develops the theory of PLS and shows precisely when and why it differs from Ridge.

## 1.2 The PLS Objective: Maximizing Covariance

Assume $\mathbf{X}$ and $\mathbf{y}$ are centered (mean-subtracted). PLS finds successive **weight vectors** $\mathbf{w}_1, \mathbf{w}_2, \ldots$ that define latent directions in feature space.

### The first PLS component

The first weight vector solves:

$$\mathbf{w}_1 = \arg\max_{\|\mathbf{w}\| = 1} \; \text{Cov}^2(\mathbf{X}\mathbf{w},\; \mathbf{y})$$

Since $\mathbf{X}$ and $\mathbf{y}$ are centered, $\text{Cov}(\mathbf{X}\mathbf{w}, \mathbf{y}) = \frac{1}{n-1}\mathbf{w}^T\mathbf{X}^T\mathbf{y}$, so we are maximizing:

$$(\mathbf{w}^T \mathbf{X}^T \mathbf{y})^2 \quad \text{subject to} \quad \|\mathbf{w}\| = 1$$

By the **Cauchy-Schwarz inequality**, $(\mathbf{w}^T \mathbf{z})^2 \leq \|\mathbf{w}\|^2 \|\mathbf{z}\|^2$ with equality when $\mathbf{w} \propto \mathbf{z}$. Setting $\mathbf{z} = \mathbf{X}^T\mathbf{y}$:

$$\boxed{\mathbf{w}_1 = \frac{\mathbf{X}^T\mathbf{y}}{\|\mathbf{X}^T\mathbf{y}\|}}$$

This is beautifully simple: the first PLS direction is proportional to the **marginal regression** of $\mathbf{y}$ on each feature. Features that are individually correlated with $\mathbf{y}$ receive large weight.

### Contrast with PCA

PCA finds directions maximizing $\text{Var}(\mathbf{X}\mathbf{w}) = \mathbf{w}^T \mathbf{X}^T\mathbf{X} \mathbf{w}$, i.e., the **variance** of the projection, ignoring $\mathbf{y}$ entirely. The leading PCA direction is the top eigenvector of $\mathbf{X}^T\mathbf{X}$.

**When PCA fails as a regression preprocessing step:** If the direction of greatest variance in $\mathbf{X}$ is *orthogonal* to the signal relevant to $\mathbf{y}$, PCA regression with few components will discard the predictive information.

> **Example.** Suppose $\mathbf{X}$ has 100 features. Features 1–99 are driven by a high-variance nuisance factor (say, batch effects), while feature 100 alone predicts $\mathbf{y}$. PCA's first component captures the nuisance; PLS's first component captures feature 100 (because it has the highest covariance with $\mathbf{y}$).

PLS is therefore a form of **supervised dimensionality reduction**: it finds low-dimensional projections that are predictive, not just high-variance.

## 1.3 The NIPALS Algorithm

PLS extracts components sequentially using deflation. The standard algorithm for PLS1 (univariate $\mathbf{y}$) is called **NIPALS** (Nonlinear Iterative Partial Least Squares):

---

**Input:** Centered $\mathbf{X} \in \mathbb{R}^{n \times p}$, centered $\mathbf{y} \in \mathbb{R}^n$, number of components $k$.

**For** $a = 1, \ldots, k$:

1. **Weight vector:** $\;\mathbf{w}_a = \mathbf{X}^T \mathbf{y} \;/\; \|\mathbf{X}^T \mathbf{y}\|$

2. **Score vector:** $\;\mathbf{t}_a = \mathbf{X} \mathbf{w}_a$

3. **X-loading:** $\;\mathbf{p}_a = \mathbf{X}^T \mathbf{t}_a \;/\; (\mathbf{t}_a^T \mathbf{t}_a)$

4. **Y-loading:** $\;q_a = \mathbf{y}^T \mathbf{t}_a \;/\; (\mathbf{t}_a^T \mathbf{t}_a)$

5. **Deflate:** $\;\mathbf{X} \leftarrow \mathbf{X} - \mathbf{t}_a \mathbf{p}_a^T$, $\;\;\mathbf{y} \leftarrow \mathbf{y} - q_a \mathbf{t}_a$

---

**Why deflation?** After extracting component $a$, we remove its contribution from both $\mathbf{X}$ and $\mathbf{y}$ so that the next component captures *new* covariance structure. Without deflation, every component would be identical to the first.

### Recovery of regression coefficients

After extracting all $k$ components, collect:
- $\mathbf{W} = [\mathbf{w}_1, \ldots, \mathbf{w}_k] \in \mathbb{R}^{p \times k}$ (weights)
- $\mathbf{P} = [\mathbf{p}_1, \ldots, \mathbf{p}_k] \in \mathbb{R}^{p \times k}$ (X-loadings)
- $\mathbf{q} = [q_1, \ldots, q_k]^T \in \mathbb{R}^k$ (Y-loadings)

The regression coefficient vector is:

$$\boxed{\hat{\boldsymbol{\beta}}_{\text{PLS}} = \mathbf{W}(\mathbf{P}^T\mathbf{W})^{-1}\mathbf{q}}$$

The matrix $\mathbf{R} = \mathbf{W}(\mathbf{P}^T\mathbf{W})^{-1}$ is sometimes called the **rotation matrix**; the columns of $\mathbf{X}\mathbf{R}$ give orthogonal score vectors.

### Prediction on new data

For centered training data with means $\bar{\mathbf{x}}$ and $\bar{y}$:

$$\hat{y}_{\text{new}} = (\mathbf{x}_{\text{new}} - \bar{\mathbf{x}})^T \hat{\boldsymbol{\beta}}_{\text{PLS}} + \bar{y}$$

## 1.4 How PLS and Ridge Regularize: A Geometric Comparison

Both PLS and Ridge can be understood through the **Singular Value Decomposition** of $\mathbf{X}$. Write $\mathbf{X} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T$, where $\boldsymbol{\Sigma} = \text{diag}(\sigma_1, \ldots, \sigma_r)$ with $r = \text{rank}(\mathbf{X})$.

### Ridge filter factors

The Ridge solution is:
$$\hat{\boldsymbol{\beta}}_{\text{ridge}} = \mathbf{V} \text{diag}\!\left(\frac{\sigma_j}{\sigma_j^2 + \lambda}\right) \mathbf{U}^T \mathbf{y}$$

This applies a **filter factor** $f_j^{\text{ridge}} = \frac{\sigma_j^2}{\sigma_j^2 + \lambda}$ to each SVD component. Key properties:
- $f_j \in [0, 1]$ for all $j$
- $f_j$ is **monotonically increasing** in $\sigma_j$: large-singular-value directions are preserved; small ones are shrunk
- Ridge treats all directions symmetrically in the SVD basis — it never *selects* directions based on their relationship with $\mathbf{y}$

**Geometric picture:** Ridge "uniformly dampens" low-variance directions, regardless of whether they carry signal about $\mathbf{y}$.

### PLS filter factors

PLS also applies filter factors in the SVD basis, but they are **not monotone** in the singular values. A PLS direction is retained if it has high covariance with $\mathbf{y}$, even if its singular value is small. Conversely, a high-variance direction that is orthogonal to $\mathbf{y}$ may receive a filter factor near zero.

| Property | Ridge | PLS ($k$ components) |
|----------|-------|---------------------|
| Filter factors monotone in $\sigma_j$ | Yes | **No** |
| Uses $\mathbf{y}$ information | No (only $\mathbf{X}$) | **Yes** |
| Effective dimensionality | Continuous (via $\lambda$) | Discrete (integer $k$) |
| Unregularized limit | $\lambda \to 0$ gives OLS | $k = \min(n, p)$ gives OLS |
| Computational cost | $O(p^2 n)$ (or $O(n^2 p)$) | $O(knp)$ |

### When does this matter?

Consider a data-generating process where $\mathbf{y}$ depends on $\mathbf{X}$ through $k_{\text{true}} \ll p$ latent factors:

$$\mathbf{X} = \mathbf{T}\mathbf{P}^T + \mathbf{E}, \qquad \mathbf{y} = \mathbf{T}\boldsymbol{\gamma} + \epsilon$$

Here $\mathbf{T} \in \mathbb{R}^{n \times k_{\text{true}}}$ are latent scores, $\mathbf{P}$ maps them to observed features, and $\boldsymbol{\gamma}$ gives the latent regression weights.

- **PLS** with $k = k_{\text{true}}$ components recovers the signal space almost exactly. It achieves low bias (captures the true subspace) and low variance (only $k_{\text{true}}$ parameters to estimate).
- **Ridge** must estimate all $p$ coefficients, merely shrinking them. It pays a variance cost for the $p - k_{\text{true}}$ noise directions, even though they carry no signal.

**Bottom line:** PLS has a strong *inductive bias* that the signal is low-rank. When this matches reality, PLS wins. When the true coefficient vector is sparse (many zeros, no latent structure), PLS's whole-subspace approach isn't as effective as Ridge or Lasso.

## 1.5 The $p \gg n$ Regime

When $p > n$, the matrix $\mathbf{X}^T\mathbf{X} \in \mathbb{R}^{p \times p}$ has rank at most $n < p$, so it is **singular**. OLS requires inverting $\mathbf{X}^T\mathbf{X}$, which is impossible.

**How Ridge resolves this.** Ridge replaces $\mathbf{X}^T\mathbf{X}$ with $\mathbf{X}^T\mathbf{X} + \lambda\mathbf{I}$. Since $\lambda > 0$, all eigenvalues are shifted above zero: if $\sigma_j^2$ was an eigenvalue of $\mathbf{X}^T\mathbf{X}$, the new eigenvalue is $\sigma_j^2 + \lambda > 0$. The matrix is always invertible.

**How PLS resolves this.** PLS never inverts $\mathbf{X}^T\mathbf{X}$ at all. Each NIPALS step only uses matrix-vector products $\mathbf{X}^T\mathbf{v}$ and $\mathbf{X}\mathbf{w}$, which are well-defined regardless of the rank of $\mathbf{X}$. The implicit regularization comes from truncating at $k < \text{rank}(\mathbf{X})$ components.

This makes PLS especially natural for spectroscopy, genomics, neuroimaging, and other domains where $p$ can be in the thousands or millions while $n$ is in the tens or hundreds.

## 1.6 Summary: When to Use PLS vs. Ridge

| Scenario | Recommended Method | Why |
|----------|-------------------|-----|
| Low-rank latent structure, $p \gg n$ | **PLS** | PLS's inductive bias matches the data; reduces effective dimension to $k$ |
| Sparse true coefficients | **Lasso** (or Ridge) | PLS doesn't perform variable selection; Lasso's $\ell_1$ penalty does |
| Collinear feature groups | **PLS** (often) | Collinear groups create latent factors; PLS extracts them naturally |
| Abundant data ($n \gg p$) | **Ridge ≈ PLS** | Both converge to OLS; regularization matters less |
| No prior knowledge of structure | **Try both + CV** | Let cross-validation decide |

With this theoretical foundation, you're ready for the computational exercises in Parts 2–6.

---
# Part 2: Implement PLS via NIPALS (20 pts)

*From here on, everything is homework. Fill in the code and written answers.*

### Problem 2.1 — NIPALS Implementation (12 pts)

Using the algorithm described in Section 1.3, implement NIPALS for PLS1 from scratch.

**Requirements:**
- Center $\mathbf{X}$ and $\mathbf{y}$ internally
- Return all components needed for prediction on new data
- Handle the intercept correctly

In [ ]:
def nipals_pls1(X, y, n_components):
    """
    PLS1 regression via the NIPALS algorithm.
    
    Parameters
    ----------
    X : ndarray of shape (n_samples, n_features)
    y : ndarray of shape (n_samples,)
    n_components : int
    
    Returns
    -------
    W : ndarray of shape (n_features, n_components) — weight vectors
    T : ndarray of shape (n_samples, n_components)  — scores
    P : ndarray of shape (n_features, n_components) — X-loadings
    q_vec : ndarray of shape (n_components,)         — Y-loadings
    beta : ndarray of shape (n_features,)            — regression coefficients
    x_mean : ndarray of shape (n_features,)
    y_mean : float
    """
    n, p = X.shape
    
    # Center
    x_mean = X.mean(axis=0)
    y_mean = y.mean()
    Xc = X - x_mean
    yc = y - y_mean
    
    # Allocate
    W = np.zeros((p, n_components))
    T = np.zeros((n, n_components))
    P = np.zeros((p, n_components))
    q_vec = np.zeros(n_components)
    
    for a in range(n_components):
        # Step 1: weight vector
        w = Xc.T @ yc
        w = w / np.linalg.norm(w)
        
        # Step 2: score vector
        t = Xc @ w
        
        # Step 3: X-loading
        p_a = Xc.T @ t / (t @ t)
        
        # Step 4: Y-loading
        q_a = yc @ t / (t @ t)
        
        # Store
        W[:, a] = w
        T[:, a] = t
        P[:, a] = p_a
        q_vec[a] = q_a
        
        # Step 5: Deflate
        Xc = Xc - np.outer(t, p_a)
        yc = yc - q_a * t
    
    # Regression coefficients: beta = W (P^T W)^{-1} q
    beta = W @ np.linalg.solve(P.T @ W, q_vec)
    
    return W, T, P, q_vec, beta, x_mean, y_mean


def predict_pls(X_new, beta, x_mean, y_mean):
    """Predict using fitted PLS model."""
    return (X_new - x_mean) @ beta + y_mean

### Problem 2.2 — Validation Against scikit-learn (8 pts)

Compare your implementation with `sklearn.cross_decomposition.PLSRegression`. They should agree to within numerical precision ($\sim 10^{-10}$).

In [ ]:
# Problem 2.2: Validate implementation
X_val, y_val, _ = generate_low_rank_data(n_samples=100, n_features=20, n_latent=3, seed=123)
X_test_val, y_test_val, _ = generate_low_rank_data(n_samples=200, n_features=20, n_latent=3, seed=456)
n_comp = 3

# Our implementation
W, T, P, q_vec, beta_mine, x_mean, y_mean = nipals_pls1(X_val, y_val, n_comp)
y_pred_mine = predict_pls(X_test_val, beta_mine, x_mean, y_mean)

# sklearn implementation
pls_sk = PLSRegression(n_components=n_comp, scale=False)
pls_sk.fit(X_val, y_val)
y_pred_sk = pls_sk.predict(X_test_val).ravel()
beta_sk = pls_sk.coef_.ravel()

print(f"Max |prediction difference|:  {np.max(np.abs(y_pred_mine - y_pred_sk)):.2e}")
print(f"Max |coefficient difference|: {np.max(np.abs(beta_mine - beta_sk)):.2e}")

In [ ]:
# Helper function for cross-validated model selection (used in multiple problems)
def cv_tune_pls_ridge(X_train, y_train, pls_range=None, ridge_range=None, n_splits=5):
    """Tune PLS n_components and Ridge alpha by CV. Returns best models fitted on full data."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)
    
    if pls_range is None:
        pls_range = np.arange(1, min(31, X_train.shape[0]))
    if ridge_range is None:
        ridge_range = np.logspace(-3, 3, 50)
    
    # PLS
    best_pls_score = -np.inf
    best_pls_k = 1
    for k in pls_range:
        pls = PLSRegression(n_components=int(k), scale=False)
        scores = cross_val_score(pls, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
        if scores.mean() > best_pls_score:
            best_pls_score = scores.mean()
            best_pls_k = int(k)
    
    pls_final = PLSRegression(n_components=best_pls_k, scale=False)
    pls_final.fit(X_train, y_train)
    
    # Ridge
    best_ridge_score = -np.inf
    best_ridge_alpha = 1.0
    for alpha in ridge_range:
        rdg = Ridge(alpha=alpha)
        scores = cross_val_score(rdg, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
        if scores.mean() > best_ridge_score:
            best_ridge_score = scores.mean()
            best_ridge_alpha = alpha
    
    ridge_final = Ridge(alpha=best_ridge_alpha)
    ridge_final.fit(X_train, y_train)
    
    return pls_final, ridge_final, best_pls_k, best_ridge_alpha

---
# Part 3: PLS vs Ridge in the $p \gg n$ Regime (25 pts)

The theory in Section 1.4 predicts that PLS should outperform Ridge when the data has low-rank latent structure and samples are scarce. Let's test this.

### Problem 3.1 — Varying the True Latent Rank (15 pts)

**Setup:**
- `generate_low_rank_data`: $n = 50$ train, $p = 200$, $n_{\text{test}} = 500$
- Vary `n_latent` $\in \{2, 5, 10, 20\}$
- CV-tune PLS (`n_components` from 1–30) and Ridge ($\alpha$ from `np.logspace(-3, 3, 50)`)
- 20 random seeds; report mean $\pm$ standard error

**Produce two plots:**
1. Test MSE vs. `n_latent`
2. Relative coefficient MSE vs. `n_latent`

**Discuss:** When the latent rank is small, which method wins? What happens as it grows?

In [ ]:
# Problem 3.1: PLS vs Ridge — Varying latent rank
# ============================================================
n_train = 50
n_test = 500
n_features = 200
latent_ranks = [2, 5, 10, 20]
n_seeds = 20

results_pls_mse = np.zeros((len(latent_ranks), n_seeds))
results_ridge_mse = np.zeros((len(latent_ranks), n_seeds))
results_pls_rel = np.zeros((len(latent_ranks), n_seeds))
results_ridge_rel = np.zeros((len(latent_ranks), n_seeds))

for i, n_latent in enumerate(latent_ranks):
    for s in range(n_seeds):
        X_tr, y_tr, beta_true = generate_low_rank_data(n_train, n_features, n_latent, seed=s)
        X_te, y_te, _ = generate_low_rank_data(n_test, n_features, n_latent, seed=s + 1000)
        
        pls_model, ridge_model, _, _ = cv_tune_pls_ridge(X_tr, y_tr)
        
        y_pls = pls_model.predict(X_te).ravel()
        y_rdg = ridge_model.predict(X_te)
        
        results_pls_mse[i, s] = mse(y_te, y_pls)
        results_ridge_mse[i, s] = mse(y_te, y_rdg)
        results_pls_rel[i, s] = relative_mse(pls_model.coef_.ravel(), beta_true)
        results_ridge_rel[i, s] = relative_mse(ridge_model.coef_, beta_true)
    print(f"n_latent={n_latent}: PLS MSE={results_pls_mse[i].mean():.2f}, Ridge MSE={results_ridge_mse[i].mean():.2f}")

In [ ]:
# Problem 3.1: Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data_pls, data_ridge, ylabel, title in [
    (axes[0], results_pls_mse, results_ridge_mse, 'Test MSE', 'Test MSE vs Latent Rank'),
    (axes[1], results_pls_rel, results_ridge_rel, 'Relative Coefficient MSE', 'Coefficient Recovery vs Latent Rank'),
]:
    mean_p = data_pls.mean(axis=1)
    se_p = data_pls.std(axis=1) / np.sqrt(n_seeds)
    mean_r = data_ridge.mean(axis=1)
    se_r = data_ridge.std(axis=1) / np.sqrt(n_seeds)
    
    ax.errorbar(latent_ranks, mean_p, yerr=se_p, marker='o', capsize=4, label='PLS')
    ax.errorbar(latent_ranks, mean_r, yerr=se_r, marker='s', capsize=4, label='Ridge')
    ax.set_xlabel('True latent rank')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()

plt.suptitle('Problem 3.1: n=50, p=200', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Discussion (3.1):**

When the true latent rank is small (e.g., 2 or 5), **PLS significantly outperforms Ridge** in both test MSE and coefficient recovery. This is because PLS's inductive bias — that the signal lies in a low-dimensional subspace — matches the data-generating process. PLS effectively reduces the problem to estimating only $k$ parameters, while Ridge must shrink all 200 coefficients.

As `n_latent` grows toward $n = 50$, the gap narrows. With 20 true latent factors, the "low-rank" assumption weakens: PLS needs more components, estimating more parameters and increasing variance. Ridge's uniform shrinkage becomes relatively more competitive because there's no strong low-rank structure to exploit.

At the limit ($k \to n$), both methods converge to similar performance — neither has a structural advantage when the true dimensionality approaches the sample size.

### Problem 3.2 — Learning Curves: Varying Sample Size (10 pts)

Fix `n_latent = 5`, $p = 200$. Vary $n \in \{20, 30, 50, 75, 100, 200\}$.

**Plot:** Test MSE vs. $n$ for both methods.

**Discuss:** At what sample size does Ridge start matching PLS? Why (bias-variance)?

In [ ]:
# Problem 3.2: Learning curves
sample_sizes = [20, 30, 50, 75, 100, 200]
n_latent_fixed = 5
n_features_3 = 200
n_test_3 = 500
n_seeds_3 = 20

results_pls_3 = np.zeros((len(sample_sizes), n_seeds_3))
results_ridge_3 = np.zeros((len(sample_sizes), n_seeds_3))

for i, n_tr in enumerate(sample_sizes):
    for s in range(n_seeds_3):
        X_tr, y_tr, _ = generate_low_rank_data(n_tr, n_features_3, n_latent_fixed, seed=s)
        X_te, y_te, _ = generate_low_rank_data(n_test_3, n_features_3, n_latent_fixed, seed=s + 1000)
        
        pls_range = np.arange(1, min(31, n_tr))
        pls_model, ridge_model, _, _ = cv_tune_pls_ridge(X_tr, y_tr, pls_range=pls_range)
        
        results_pls_3[i, s] = mse(y_te, pls_model.predict(X_te).ravel())
        results_ridge_3[i, s] = mse(y_te, ridge_model.predict(X_te))
    print(f"n={n_tr}: PLS={results_pls_3[i].mean():.2f}, Ridge={results_ridge_3[i].mean():.2f}")

In [ ]:
# Problem 3.2: Plotting
fig, ax = plt.subplots(figsize=(10, 6))
mean_p = results_pls_3.mean(axis=1)
se_p = results_pls_3.std(axis=1) / np.sqrt(n_seeds_3)
mean_r = results_ridge_3.mean(axis=1)
se_r = results_ridge_3.std(axis=1) / np.sqrt(n_seeds_3)

ax.errorbar(sample_sizes, mean_p, yerr=se_p, marker='o', capsize=4, label='PLS (CV-tuned)')
ax.errorbar(sample_sizes, mean_r, yerr=se_r, marker='s', capsize=4, label='Ridge (CV-tuned)')
ax.set_xlabel('Training sample size n')
ax.set_ylabel('Test MSE')
ax.set_title('Learning Curves: PLS vs Ridge (p=200, true latent rank=5)')
ax.legend()
plt.tight_layout()
plt.show()

**Discussion (3.2):**

1. Ridge begins to match PLS around $n \approx 100$–$200$, when the sample size becomes large enough relative to $p$ that Ridge's shrinkage can effectively estimate all coefficients.

2. **Bias-variance perspective:** PLS with $k = 5$ components only estimates $\sim 5$ effective parameters, so its variance is low even at tiny $n$. Its bias is also low because the true model *is* 5-dimensional. Ridge must estimate (a shrunk version of) all 200 coefficients — high variance at small $n$, but this decreases as $n$ grows. Once $n$ is large enough that Ridge's variance penalty is small, its minimal bias (it doesn't discard any directions) matches PLS. Eventually both approach the irreducible noise floor.

---
# Part 4: PLS vs Ridge Under Sparse Signals (25 pts)

Part 3 confirmed PLS's advantage on low-rank data. Now we flip the script: the true signal is **sparse** (few non-zero coefficients), with no latent structure.

### Problem 4.1 — Sparse Coefficients (15 pts)

**Setup:**
- `generate_sparse_data`: $n = 60$, $p = 150$
- Vary `n_informative` $\in \{3, 10, 30, 60\}$
- Two correlation settings: $\rho = 0.0$ and $\rho = 0.7$
- Include **Lasso** as a reference (sparse recovery is its strength)
- 20 seeds, test set of 500

**Create a 2×2 figure:**
- Rows: correlation setting
- Left column: grouped bar chart of test MSE (PLS / Ridge / Lasso)
- Right column: stem plots of true vs estimated coefficients (for `n_informative=10`, seed=0)

**Discuss:** Which method wins when the signal is sparse? How does correlation change things?

In [ ]:
# Problem 4.1: Sparse signals
n_train_4 = 60
n_features_4 = 150
n_test_4 = 500
informative_counts = [3, 10, 30, 60]
correlations = [0.0, 0.7]
n_seeds_4 = 20
lasso_alpha_range = np.logspace(-3, 1, 50)

# results[corr_idx]['method'] = (len(informative_counts), n_seeds_4)
results_4 = {}
saved_coefs = {}

for ci, corr in enumerate(correlations):
    res = {'pls': np.zeros((len(informative_counts), n_seeds_4)),
           'ridge': np.zeros((len(informative_counts), n_seeds_4)),
           'lasso': np.zeros((len(informative_counts), n_seeds_4))}
    
    for ii, n_inf in enumerate(informative_counts):
        for s in range(n_seeds_4):
            X_tr, y_tr, beta_true = generate_sparse_data(
                n_train_4, n_features_4, n_inf, correlation=corr, seed=s)
            X_te, y_te, _ = generate_sparse_data(
                n_test_4, n_features_4, n_inf, correlation=corr, seed=s + 1000)
            
            pls_range = np.arange(1, min(31, n_train_4))
            pls_m, ridge_m, _, _ = cv_tune_pls_ridge(X_tr, y_tr, pls_range=pls_range)
            
            # Lasso CV
            kf = KFold(n_splits=5, shuffle=True, random_state=0)
            best_lasso_score = -np.inf
            best_lasso_alpha = 0.01
            for alpha in lasso_alpha_range:
                lasso = Lasso(alpha=alpha, max_iter=10000)
                scores = cross_val_score(lasso, X_tr, y_tr, cv=kf, scoring='neg_mean_squared_error')
                if scores.mean() > best_lasso_score:
                    best_lasso_score = scores.mean()
                    best_lasso_alpha = alpha
            lasso_m = Lasso(alpha=best_lasso_alpha, max_iter=10000)
            lasso_m.fit(X_tr, y_tr)
            
            res['pls'][ii, s] = mse(y_te, pls_m.predict(X_te).ravel())
            res['ridge'][ii, s] = mse(y_te, ridge_m.predict(X_te))
            res['lasso'][ii, s] = mse(y_te, lasso_m.predict(X_te))
            
            # Save representative coefficients
            if s == 0 and n_inf == 10:
                saved_coefs[(corr, 'true')] = beta_true
                saved_coefs[(corr, 'pls')] = pls_m.coef_.ravel()
                saved_coefs[(corr, 'ridge')] = ridge_m.coef_
                saved_coefs[(corr, 'lasso')] = lasso_m.coef_
        
        print(f"ρ={corr}, s={n_inf:2d}: PLS={res['pls'][ii].mean():.2f}, "
              f"Ridge={res['ridge'][ii].mean():.2f}, Lasso={res['lasso'][ii].mean():.2f}")
    results_4[corr] = res

In [ ]:
# Problem 4.1: 2x2 figure
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ci, corr in enumerate(correlations):
    res = results_4[corr]
    ax_bar = axes[ci, 0]
    ax_stem = axes[ci, 1]
    
    # Bar chart
    x = np.arange(len(informative_counts))
    width = 0.25
    for mi, (method, color) in enumerate([('pls', 'C0'), ('ridge', 'C1'), ('lasso', 'C2')]):
        means = res[method].mean(axis=1)
        ses = res[method].std(axis=1) / np.sqrt(n_seeds_4)
        ax_bar.bar(x + mi * width, means, width, yerr=ses, capsize=3, 
                   label=method.upper(), color=color, alpha=0.8)
    ax_bar.set_xticks(x + width)
    ax_bar.set_xticklabels(informative_counts)
    ax_bar.set_xlabel('# informative features')
    ax_bar.set_ylabel('Test MSE')
    ax_bar.set_title(f'Sparse signal (ρ = {corr}): Test MSE')
    ax_bar.legend()
    
    # Stem plot (n_informative=10)
    idx = np.arange(n_features_4)
    beta_true = saved_coefs[(corr, 'true')]
    ax_stem.stem(idx, beta_true, linefmt='k-', markerfmt='ko', basefmt='k-', label='True', use_line_collection=True)
    for method, color, marker in [('pls', 'C0', '^'), ('ridge', 'C1', 's'), ('lasso', 'C2', 'D')]:
        beta_hat = saved_coefs[(corr, method)]
        ax_stem.scatter(idx, beta_hat, c=color, s=8, alpha=0.6, label=method.upper(), marker=marker)
    ax_stem.set_xlabel('Feature index')
    ax_stem.set_ylabel('Coefficient')
    ax_stem.set_title(f'Coefficients (ρ = {corr}, s = 10)')
    ax_stem.legend(fontsize=8)
    ax_stem.axhline(0, color='gray', lw=0.5)

plt.tight_layout()
plt.show()

**Discussion (4.1):**

1. **Lasso wins** when the signal is truly sparse (small `n_informative`), because its $\ell_1$ penalty performs variable selection. Ridge comes second; PLS typically performs worst. PLS constructs latent components as linear combinations of *all* features — it cannot zero out irrelevant ones the way Lasso does, and its whole-subspace approach doesn't match the sparse generating process.

2. **With correlated features** ($\rho = 0.7$), PLS closes the gap somewhat. Correlation creates implicit low-rank structure: correlated features form quasi-groups, and PLS can extract these group-level signals. Lasso's advantage also shrinks because it struggles to select among correlated predictors (it tends to pick one arbitrarily).

3. PLS performs comparably to Ridge when `n_informative` is large (many nonzero coefficients) — the signal is no longer truly sparse, and the effective structure looks more "diffuse," which is closer to what PLS handles. With high correlation, PLS can even beat Ridge because the correlation structure provides latent factors to exploit.

### Problem 4.2 — Collinear Groups (10 pts)

Features in groups of collinear variables: a setting *between* low-rank and sparse.

**Setup:** `generate_collinear_data`: $n = 60$, $p = 100$, 5 groups of 10 + 50 noise features.

**Produce:** Bar chart of test MSE + stem plots showing coefficient recovery. Highlight group boundaries.

**Discuss:** Which method better recovers the group structure?

In [ ]:
# Problem 4.2: Collinear groups
n_train_42 = 60
n_features_42 = 100
n_groups = 5
group_size = 10
n_test_42 = 500
n_seeds_42 = 20

results_pls_42 = np.zeros(n_seeds_42)
results_ridge_42 = np.zeros(n_seeds_42)
saved_42 = {}

for s in range(n_seeds_42):
    X_tr, y_tr, beta_true = generate_collinear_data(n_train_42, n_features_42, n_groups, group_size, seed=s)
    X_te, y_te, _ = generate_collinear_data(n_test_42, n_features_42, n_groups, group_size, seed=s + 1000)
    
    pls_range = np.arange(1, min(21, n_train_42))
    pls_m, ridge_m, _, _ = cv_tune_pls_ridge(X_tr, y_tr, pls_range=pls_range)
    
    results_pls_42[s] = mse(y_te, pls_m.predict(X_te).ravel())
    results_ridge_42[s] = mse(y_te, ridge_m.predict(X_te))
    
    if s == 0:
        saved_42['true'] = beta_true
        saved_42['pls'] = pls_m.coef_.ravel()
        saved_42['ridge'] = ridge_m.coef_

print(f"PLS:   {results_pls_42.mean():.3f} ± {results_pls_42.std()/np.sqrt(n_seeds_42):.3f}")
print(f"Ridge: {results_ridge_42.mean():.3f} ± {results_ridge_42.std()/np.sqrt(n_seeds_42):.3f}")

In [ ]:
# Problem 4.2: Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart
means = [results_pls_42.mean(), results_ridge_42.mean()]
ses = [results_pls_42.std() / np.sqrt(n_seeds_42), results_ridge_42.std() / np.sqrt(n_seeds_42)]
axes[0].bar(['PLS', 'Ridge'], means, yerr=ses, capsize=5, color=['C0', 'C1'], alpha=0.8)
axes[0].set_ylabel('Test MSE')
axes[0].set_title('Collinear Groups: Test MSE')

# Coefficient plots
for ax, method, title in [(axes[1], 'pls', 'PLS'), (axes[2], 'ridge', 'Ridge')]:
    idx = np.arange(n_features_42)
    ax.stem(idx, saved_42['true'], linefmt='k-', markerfmt='ko', basefmt=' ', label='True', use_line_collection=True)
    color = 'C0' if method == 'pls' else 'C1'
    ax.scatter(idx, saved_42[method], c=color, s=12, alpha=0.7, label=f'{title} estimate', zorder=3)
    for b in [10, 20, 30, 40, 50]:
        ax.axvline(b - 0.5, color='red', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xlabel('Feature index')
    ax.set_ylabel('Coefficient')
    ax.set_title(f'{title}: Coefficient Recovery')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

**Discussion (4.2):**

PLS tends to better recover the group structure of the coefficients. Within each group, the features share a common latent factor — exactly the kind of structure PLS is designed to detect. PLS extracts group-level latent components, assigning similar weights to features within the same group and near-zero weights to noise features.

Ridge, by contrast, shrinks all coefficients uniformly toward zero and doesn't "recognize" groups. It spreads weight more diffusely across both informative and noise features. The resulting coefficient profile is flatter and noisier.

This setting is favorable for PLS because collinear groups *create* approximate low-rank structure. Each group contributes one effective latent factor, and PLS with ~5 components should capture all of them efficiently.

---
# Part 5: Hyperparameter Sensitivity and Component Visualization (15 pts)

### Problem 5.1 — PLS Components vs Ridge $\alpha$: A CV Study (10 pts)

For a fixed setting ($n = 50$, $p = 100$, `n_latent = 3`, seed = 42):
1. Sweep PLS components 1–20: compute 5-fold CV MSE and test MSE
2. Sweep Ridge $\alpha$ over `np.logspace(-3, 5, 80)`: same
3. Plot side-by-side, marking CV-optimal points

**Discuss:** Which method's optimum is sharper? Which is more robust to hyperparameter mis-specification?

In [ ]:
# Problem 5.1: Hyperparameter sensitivity
n_train_5 = 50
n_features_5 = 100
n_latent_5 = 3
seed_5 = 42

X_train_5, y_train_5, beta_true_5 = generate_low_rank_data(n_train_5, n_features_5, n_latent_5, seed=seed_5)
X_test_5, y_test_5, _ = generate_low_rank_data(500, n_features_5, n_latent_5, seed=seed_5 + 1000)

pls_comp_range_5 = np.arange(1, 21)
ridge_alpha_range_5 = np.logspace(-3, 5, 80)
kf5 = KFold(n_splits=5, shuffle=True, random_state=0)

pls_cv_mse = []
pls_test_mse = []
for k in pls_comp_range_5:
    pls = PLSRegression(n_components=int(k), scale=False)
    scores = cross_val_score(pls, X_train_5, y_train_5, cv=kf5, scoring='neg_mean_squared_error')
    pls_cv_mse.append(-scores.mean())
    pls.fit(X_train_5, y_train_5)
    pls_test_mse.append(mse(y_test_5, pls.predict(X_test_5).ravel()))

ridge_cv_mse = []
ridge_test_mse = []
for alpha in ridge_alpha_range_5:
    rdg = Ridge(alpha=alpha)
    scores = cross_val_score(rdg, X_train_5, y_train_5, cv=kf5, scoring='neg_mean_squared_error')
    ridge_cv_mse.append(-scores.mean())
    rdg.fit(X_train_5, y_train_5)
    ridge_test_mse.append(mse(y_test_5, rdg.predict(X_test_5)))

In [ ]:
# Problem 5.1: Side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PLS
best_k = pls_comp_range_5[np.argmin(pls_cv_mse)]
axes[0].plot(pls_comp_range_5, pls_cv_mse, 'b-o', label='CV MSE', markersize=5)
axes[0].plot(pls_comp_range_5, pls_test_mse, 'r--s', label='Test MSE', markersize=5)
axes[0].axvline(best_k, color='green', ls=':', lw=2, label=f'CV-optimal k={best_k}')
axes[0].set_xlabel('Number of PLS components')
axes[0].set_ylabel('MSE')
axes[0].set_title('PLS: CV vs Test MSE')
axes[0].legend()

# Ridge
best_alpha = ridge_alpha_range_5[np.argmin(ridge_cv_mse)]
axes[1].plot(np.log10(ridge_alpha_range_5), ridge_cv_mse, 'b-o', label='CV MSE', markersize=3)
axes[1].plot(np.log10(ridge_alpha_range_5), ridge_test_mse, 'r--s', label='Test MSE', markersize=3)
axes[1].axvline(np.log10(best_alpha), color='green', ls=':', lw=2, label=f'CV-optimal α={best_alpha:.1e}')
axes[1].set_xlabel('log₁₀(α)')
axes[1].set_ylabel('MSE')
axes[1].set_title('Ridge: CV vs Test MSE')
axes[1].legend()

plt.tight_layout()
plt.show()

**Discussion (5.1):**

PLS's optimum is typically **sharper** — performance degrades quickly if you use too many components (overfitting) or too few (underfitting). The optimal number of components (near the true latent rank) sits in a narrow valley. Adding one component beyond the true rank often causes a noticeable jump in test error because the extra component overfits noise.

Ridge's optimum is **broader**: a wide range of $\alpha$ values near the optimal point give similar performance. The continuous nature of $\alpha$ and the smooth shrinkage it applies make Ridge more robust to hyperparameter mis-specification.

**Practical implication:** Ridge is more "forgiving" — even a rough grid search yields near-optimal performance. PLS requires more careful CV because the integer constraint on components means each step can cause a larger performance change.

### Problem 5.2 — PLS vs PCA Latent Components (5 pts)

Using the same dataset:
1. Extract 3 PLS scores and 3 PCA scores
2. Scatter-plot grid (2×3), color-coded by $\mathbf{y}$
3. Report correlation of each component with $\mathbf{y}$

**Discuss:** Which components are more aligned with the response?

In [ ]:
# Problem 5.2: PLS vs PCA components
pls_vis = PLSRegression(n_components=3, scale=False)
pls_vis.fit(X_train_5, y_train_5)
pls_scores = pls_vis.x_scores_

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_5)
pca_vis = PCA(n_components=3)
pca_scores = pca_vis.fit_transform(X_scaled)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
pairs = [(0, 1), (0, 2), (1, 2)]
for col, (i, j) in enumerate(pairs):
    sc0 = axes[0, col].scatter(pls_scores[:, i], pls_scores[:, j], c=y_train_5, cmap='viridis', s=25)
    axes[0, col].set_xlabel(f'PLS component {i+1}')
    axes[0, col].set_ylabel(f'PLS component {j+1}')
    
    sc1 = axes[1, col].scatter(pca_scores[:, i], pca_scores[:, j], c=y_train_5, cmap='viridis', s=25)
    axes[1, col].set_xlabel(f'PCA component {i+1}')
    axes[1, col].set_ylabel(f'PCA component {j+1}')

axes[0, 1].set_title('PLS Scores (colored by y)', fontsize=13)
axes[1, 1].set_title('PCA Scores (colored by y)', fontsize=13)
plt.colorbar(sc0, ax=axes[0, :], label='y', shrink=0.6)
plt.colorbar(sc1, ax=axes[1, :], label='y', shrink=0.6)
plt.tight_layout()
plt.show()

print("Correlation of each component with y:")
print(f"{'Component':<12} {'PLS':>8} {'PCA':>8}")
for k in range(3):
    r_pls = np.corrcoef(pls_scores[:, k], y_train_5)[0, 1]
    r_pca = np.corrcoef(pca_scores[:, k], y_train_5)[0, 1]
    print(f"  {k+1:<10} {r_pls:>8.3f} {r_pca:>8.3f}")

**Discussion (5.2):**

PLS components show dramatically clearer alignment with $\mathbf{y}$. The first PLS component typically has $|\text{corr}(t_1, y)| > 0.9$, while the first PCA component may have a much weaker (or even near-zero) correlation with $y$.

This is expected by construction: PLS *maximizes covariance between $\mathbf{X}\mathbf{w}$ and $\mathbf{y}$*, so each component is explicitly designed to capture predictive signal. PCA maximizes variance in $\mathbf{X}$ alone. If the highest-variance directions in $\mathbf{X}$ happen to be orthogonal to $\mathbf{y}$, PCA components will be uninformative for regression.

In the scatter plots, the PLS panels show a clear color gradient (smooth transition from blue to yellow), indicating that the components separate high vs. low $y$ values. The PCA panels show a more jumbled color pattern.

---
# Part 6: Synthesis (10 pts)

### Problem 6.1 — Summary Table and Heatmap (5 pts)

Compile results from Parts 3–4 into:
1. A summary table with test MSE for PLS and Ridge across all conditions
2. A heatmap of PLS MSE / Ridge MSE ratios (< 1 = PLS wins)

In [ ]:
# Problem 6.1: Summary table and heatmap
rows = []

# Part 3.1: Varying latent rank
for i, lr in enumerate(latent_ranks):
    pls_mean = results_pls_mse[i].mean()
    pls_se = results_pls_mse[i].std() / np.sqrt(n_seeds)
    ridge_mean = results_ridge_mse[i].mean()
    ridge_se = results_ridge_mse[i].std() / np.sqrt(n_seeds)
    winner = 'PLS' if pls_mean < ridge_mean else 'Ridge'
    rows.append({
        'Setting': f'Low-rank, k={lr}, n=50, p=200',
        'PLS MSE': f'{pls_mean:.2f} ± {pls_se:.2f}',
        'Ridge MSE': f'{ridge_mean:.2f} ± {ridge_se:.2f}',
        'Ratio': pls_mean / ridge_mean,
        'Winner': winner
    })

# Part 4.1: Sparse
for corr in correlations:
    res = results_4[corr]
    for ii, n_inf in enumerate(informative_counts):
        pls_mean = res['pls'][ii].mean()
        ridge_mean = res['ridge'][ii].mean()
        pls_se = res['pls'][ii].std() / np.sqrt(n_seeds_4)
        ridge_se = res['ridge'][ii].std() / np.sqrt(n_seeds_4)
        winner = 'PLS' if pls_mean < ridge_mean else 'Ridge'
        rows.append({
            'Setting': f'Sparse, s={n_inf}, ρ={corr}, n=60, p=150',
            'PLS MSE': f'{pls_mean:.2f} ± {pls_se:.2f}',
            'Ridge MSE': f'{ridge_mean:.2f} ± {ridge_se:.2f}',
            'Ratio': pls_mean / ridge_mean,
            'Winner': winner
        })

# Collinear
pls_mean = results_pls_42.mean()
ridge_mean = results_ridge_42.mean()
rows.append({
    'Setting': 'Collinear groups, n=60, p=100',
    'PLS MSE': f'{pls_mean:.3f} ± {results_pls_42.std()/np.sqrt(n_seeds_42):.3f}',
    'Ridge MSE': f'{ridge_mean:.3f} ± {results_ridge_42.std()/np.sqrt(n_seeds_42):.3f}',
    'Ratio': pls_mean / ridge_mean,
    'Winner': 'PLS' if pls_mean < ridge_mean else 'Ridge'
})

df_summary = pd.DataFrame(rows)
display(df_summary[['Setting', 'PLS MSE', 'Ridge MSE', 'Winner']])

# Heatmap of ratios
ratio_data = df_summary[['Setting', 'Ratio']].set_index('Setting')
fig, ax = plt.subplots(figsize=(6, 8))
sns.heatmap(ratio_data, annot=True, fmt='.2f', cmap='RdBu_r', center=1.0,
            ax=ax, cbar_kws={'label': 'PLS/Ridge MSE ratio'})
ax.set_title('PLS / Ridge MSE Ratio\n(< 1 = PLS wins, > 1 = Ridge wins)')
plt.tight_layout()
plt.show()

### Problem 6.2 — Written Analysis (5 pts)

Write a concise analysis addressing:
1. Under what conditions does PLS significantly outperform Ridge?
2. Under what conditions does Ridge match or beat PLS?
3. Bias-variance perspective on each method's regularization strategy.
4. Practical recommendation for a new dataset ($n = 80$, $p = 300$, unknown structure).

**Analysis (6.2):**

**1. When PLS wins:** PLS significantly outperforms Ridge when the data has genuine low-rank latent structure and the number of samples is small relative to features. In our experiments, PLS was clearly superior for low-rank data ($k \in \{2, 5\}$) with $n = 50, p = 200$, and also on collinear-group data where feature groups create natural latent factors. The advantage was most pronounced in the extreme small-sample regime ($n = 20$–$50$).

**2. When Ridge wins:** Ridge matches or beats PLS when (a) the true signal is sparse rather than low-rank, (b) the sample size is large enough that Ridge's uniform shrinkage can effectively estimate all coefficients, or (c) features are independent with no exploitable covariance structure. In the sparse regime with independent features, Ridge consistently outperformed PLS, and Lasso outperformed both.

**3. Bias-variance tradeoff:** PLS with $k$ components restricts predictions to a $k$-dimensional subspace, dramatically reducing variance (only ~$k$ effective parameters) at the cost of bias if the true signal doesn't lie in that subspace. Ridge shrinks all $p$ coefficients toward zero — lower bias (it doesn't discard any direction) but higher variance because all $p$ parameters remain in play, albeit dampened. In the $p \gg n$ regime, variance dominates, so PLS's aggressive dimension reduction is advantageous *if* the signal truly is low-rank.

**4. Practical recommendation:** For $n = 80, p = 300$ with unknown structure, I recommend **running both PLS and Ridge (and Lasso) with cross-validation and selecting the winner**. If domain knowledge suggests latent structure (e.g., spectroscopy, gene expression driven by pathways), start with PLS. If features are believed to be independently informative, favor Ridge or Lasso. When unsure, the CV comparison itself is diagnostic: if PLS's CV-optimal number of components is small (e.g., 3–5), that suggests exploitable latent structure.

---
# Bonus: The PLS Regularization Path (10 pts extra credit)

Plot coefficient paths for PLS (varying `n_components` from 1 to 30) and Ridge (varying $\alpha$ from high to low). Highlight the 5 features with largest true $|\beta_j|$.

**Discuss:** How do the paths differ? Does PLS show feature-selection-like behavior? Is it monotone?

In [ ]:
# Bonus: Coefficient paths
# PLS path
pls_betas = np.zeros((30, n_features_5))
for k in range(1, 31):
    pls = PLSRegression(n_components=k, scale=False)
    pls.fit(X_train_5, y_train_5)
    pls_betas[k-1] = pls.coef_.ravel()

# Ridge path
ridge_alphas = np.logspace(5, -3, 80)
ridge_betas = np.zeros((len(ridge_alphas), n_features_5))
for i, alpha in enumerate(ridge_alphas):
    rdg = Ridge(alpha=alpha)
    rdg.fit(X_train_5, y_train_5)
    ridge_betas[i] = rdg.coef_.ravel()

top5 = np.argsort(np.abs(beta_true_5))[-5:]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for j in range(n_features_5):
    color = 'red' if j in top5 else 'steelblue'
    alpha_val = 0.9 if j in top5 else 0.08
    lw = 2.5 if j in top5 else 0.4
    axes[0].plot(range(1, 31), pls_betas[:, j], color=color, alpha=alpha_val, lw=lw)
    axes[1].plot(-np.log10(ridge_alphas), ridge_betas[:, j], color=color, alpha=alpha_val, lw=lw)

axes[0].set_xlabel('Number of PLS components')
axes[0].set_ylabel('Coefficient value')
axes[0].set_title('PLS Coefficient Path')
axes[0].axhline(y=0, color='black', lw=0.5)

axes[1].set_xlabel('-log₁₀(α)  [less regularization →]')
axes[1].set_ylabel('Coefficient value')
axes[1].set_title('Ridge Coefficient Path')
axes[1].axhline(y=0, color='black', lw=0.5)

# Legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='red', lw=2.5, label='Top-5 |β_true|'),
                   Line2D([0], [0], color='steelblue', lw=1, alpha=0.4, label='Other features')]
axes[0].legend(handles=legend_elements)
axes[1].legend(handles=legend_elements)

plt.tight_layout()
plt.show()

**Discussion (Bonus):**

The paths differ strikingly:

**Ridge** grows coefficients **smoothly and monotonically** as $\alpha$ decreases (moving right). All coefficients start at zero and gradually increase in magnitude. The large-$|\beta|$ features (red) grow faster but all features grow simultaneously — Ridge never "selects" features.

**PLS** grows coefficients in **discrete jumps** (one per component added). The path is *not* monotone: individual coefficients can increase then decrease, or even change sign, as new components are added. This non-monotonicity is a key difference from Ridge.

PLS does show partial feature-selection-like behavior: after one component, only features correlated with $\mathbf{w}_1$ have nonzero coefficients (proportional to their loading). Important features tend to receive weight earlier (at fewer components). However, PLS never truly zeroes out features — it's not a sparsity method. The path's non-monotonicity arises because adding a new component can redistribute weight among features as the deflated $\mathbf{X}$ changes.

*End of solutions.*